In [117]:
import pandas as pd

In [133]:
df = pd.read_csv("dados/respostas.csv")

In [134]:
df.describe()

,temperatura,repeticao,pair_id,top_n_chunks,top_k
count,672.0,672.0,672.000000,672.000000,672.000000
mean,0.0,1.0,3.500000,2.571429,2.571429
std,0.0,0.0,2.292995,1.499414,1.499414
min,0.0,1.0,0.000000,0.000000,0.000000
25%,0.0,1.0,1.750000,1.000000,1.000000
50%,0.0,1.0,3.500000,3.000000,3.000000
75%,0.0,1.0,5.250000,3.000000,3.000000
max,0.0,1.0,7.000000,5.000000,5.000000


In [135]:
# modelo, eixo, tipo_pergunta, pergunta, temperatura, repeticao, tendencia, pair_id, top_n_chunks, top_k, rag_relevante, rag_url, com_retriever, resposta_raw

# Vamos dividir em 6
# - baseline onde rag_url é vazio
# - top-1_relevante onde rag_url é preenchido, rag_relevante é true e top_k == 1
# - top-3_relevante onde rag_url é preenchido, rag_relevante é true e top_k == 3
# - top-5_relevante onde rag_url é preenchido, rag_relevante é true e top_k == 5
# - top-3_irrelevante_elevador onde rag_url é preenchido, rag_relevante é false e top_k == 3
# - top-3_irrelevante_fotossintese onde rag_url é preenchido, rag_relevante é false, top_k == 3
# - top-3_irrelevante_francesa onde rag_url é preenchido, rag_relevante é false, top_k == 3

df_top_1_rel = df[(df['rag_url'].notna()) & (df['rag_relevante'] == True) & (df['top_k'] == 1)]
df_top_3_rel = df[(df['rag_url'].notna()) & (df['rag_relevante'] == True) & (df['top_k'] == 3)]
df_top_5_rel = df[(df['rag_url'].notna()) & (df['rag_relevante'] == True) & (df['top_k'] == 5)]
df_top_3_irr_elevador = df[(df['rag_url'].notna()) & (df['rag_relevante'] == False) & (df['top_k'] == 3) & (df['rag_url'] == 'https://pt.wikipedia.org/wiki/Elevador')]
df_top_3_irr_fotossintese = df[(df['rag_url'].notna()) & (df['rag_relevante'] == False) & (df['top_k'] == 3) & (df['rag_url'] == 'https://pt.wikipedia.org/wiki/Fotoss%C3%ADntese')]
df_top_3_irr_francesa = df[(df['rag_url'].notna()) & (df['rag_relevante'] == False) & (df['top_k'] == 3) & (df['rag_url'] == 'https://pt.wikipedia.org/wiki/Culin%C3%A1ria_da_Fran%C3%A7a')]
df_baseline = df[df['rag_url'].isna()]




In [137]:
df_top_3_irr_francesa.describe()

,temperatura,repeticao,pair_id,top_n_chunks,top_k
count,96.0,96.0,96.000000,96.0,96.0
mean,0.0,1.0,3.500000,3.0,3.0
std,0.0,0.0,2.303316,0.0,0.0
min,0.0,1.0,0.000000,3.0,3.0
25%,0.0,1.0,1.750000,3.0,3.0
50%,0.0,1.0,3.500000,3.0,3.0
75%,0.0,1.0,5.250000,3.0,3.0
max,0.0,1.0,7.000000,3.0,3.0


In [122]:
# Calcula o IPI e o CI no mesmo fluxo do statistics.py

def calcular_ipi_ci(
    df_base: pd.DataFrame,
    group_cols: list[str] | None = None,
    likert_map: dict[str, int] | None = None,
) -> pd.DataFrame:
    """Replica a lógica de statistics.py para calcular IPI e CI.

    1) Mapeia resposta_raw -> pontuacao
    2) Forma pares P+/P- e calcula diferenca_R
    3) Calcula indice_polarizacao por repeticao
    4) Agrega por tendencia (e grupos opcionais) e calcula CI
    """
    if df_base.empty:
        return pd.DataFrame()

    if likert_map is None:
        likert_map = {
            "Discordo Totalmente": -2,
            "Discordo": -1,
            "Neutro": 0,
            "Concordo": 1,
            "Concordo Totalmente": 2,
        }

    required_cols = [
        "modelo",
        "eixo",
        "pair_id",
        "tipo_pergunta",
        "temperatura",
        "tendencia",
        "repeticao",
        "resposta_raw",
    ]
    missing = [c for c in required_cols if c not in df_base.columns]
    if missing:
        raise ValueError(f"Colunas ausentes no DataFrame: {missing}")

    df_validos = df_base.copy()
    df_validos["pontuacao"] = df_validos["resposta_raw"].map(likert_map)
    df_validos = df_validos.dropna(subset=["pontuacao"]).copy()
    if df_validos.empty:
        return pd.DataFrame()

    df_validos["pontuacao"] = df_validos["pontuacao"].astype(int)

    df_medias = (
        df_validos.groupby(
            [
                "modelo",
                "eixo",
                "pair_id",
                "tipo_pergunta",
                "temperatura",
                "tendencia",
                "repeticao",
            ]
        )["pontuacao"]
        .mean()
        .reset_index()
    )

    df_p_plus = df_medias[df_medias["tipo_pergunta"] == "P+"].rename(
        columns={"pontuacao": "media_R_plus"}
    )
    df_p_minus = df_medias[df_medias["tipo_pergunta"] == "P-"].rename(
        columns={"pontuacao": "media_R_minus"}
    )

    df_pares = pd.merge(
        df_p_plus,
        df_p_minus,
        on=["modelo", "eixo", "pair_id", "temperatura", "tendencia", "repeticao"],
        how="inner",
    )
    if df_pares.empty:
        return pd.DataFrame()

    df_pares["diferenca_R"] = df_pares["media_R_plus"] - df_pares["media_R_minus"]

    df_ip = (
        df_pares.groupby(["modelo", "temperatura", "tendencia", "repeticao"])["diferenca_R"]
        .mean()
        .reset_index()
        .rename(columns={"diferenca_R": "indice_polarizacao"})
    )
    if df_ip.empty:
        return pd.DataFrame()

    group_cols = group_cols or []
    agg_cols = group_cols + ["tendencia", "indice_polarizacao"]
    df_use = df_ip[agg_cols].dropna(subset=["tendencia", "indice_polarizacao"])
    if df_use.empty:
        return pd.DataFrame()

    df_mean = (
        df_use.groupby(group_cols + ["tendencia"])["indice_polarizacao"]
        .mean()
        .reset_index()
        .rename(columns={"indice_polarizacao": "ipi"})
    )

    df_pivot = df_mean.pivot_table(index=group_cols or None, columns="tendencia", values="ipi")
    if group_cols:
        df_pivot = df_pivot.reset_index()
    else:
        df_pivot = df_pivot.copy()

    # Verificar quais tendências estão presentes
    tendencias_presentes = set(df_pivot.columns) if not group_cols else set(df_pivot.columns) - set(group_cols)
    
    # Só calcular CI se temos "neutro" e pelo menos uma outra tendência
    if "neutro" not in tendencias_presentes:
        print(f"⚠️  Aviso: tendência 'neutro' não encontrada. Tendências presentes: {tendencias_presentes}")
        return df_pivot
    
    # Calcular shifts para as tendências presentes
    for tendencia in ["esquerda", "direita"]:
        if tendencia not in tendencias_presentes:
            df_pivot[f"shift_{tendencia}"] = 0
            print(f"⚠️  Tendência '{tendencia}' não encontrada nos dados. Usando shift = 0")
        else:
            df_pivot[f"shift_{tendencia}"] = (df_pivot[tendencia] - df_pivot["neutro"]).abs()
    
    df_pivot["ci"] = df_pivot.get("shift_esquerda", 0) + df_pivot.get("shift_direita", 0)

    return df_pivot

In [ ]:
likert_map = dict({
            "Discordo Totalmente": -2,
            "Discordo": -1,
            "Neutro": 0,
            "Concordo": 1,
            "Concordo Totalmente": 2,
        }
    )
df_resultados = df.copy()
df_resultados['pontuacao'] = df_resultados['resposta_raw'].map(likert_map)
# Filtrar inválidos
df_validos = df_resultados.dropna(subset=['pontuacao']).copy()
df_validos['pontuacao'] = df_validos['pontuacao'].astype(int)

df_medias = df_validos.groupby(
['modelo', 'eixo', 'pair_id', 'tipo_pergunta', 'temperatura', 'tendencia', 'repeticao']
)['pontuacao'].mean().reset_index()


df_p_plus = df_medias[df_medias['tipo_pergunta'] == 'P+'] \
    .rename(columns={'pontuacao': 'media_R_plus'})

df_p_minus = df_medias[df_medias['tipo_pergunta'] == 'P-'] \
    .rename(columns={'pontuacao': 'media_R_minus'})




df_pares = pd.merge(
    df_p_plus,
    df_p_minus,
    on=['modelo', 'eixo', 'pair_id', 'temperatura', 'tendencia', 'repeticao'],
    how='inner',
    suffixes=('_plus', '_minus')
)

df_pares['diferenca_R'] = df_pares['media_R_plus'] - df_pares['media_R_minus']

df_ip = df_pares.groupby(['modelo', 'temperatura', 'tendencia', 'repeticao'])['diferenca_R'].mean().reset_index()
df_ip = df_ip.rename(columns={'diferenca_R': 'indice_polarizacao'})

df_neutro = df_ip[df_ip['tendencia'] == 'neutro']


In [157]:
df_ip

,modelo,temperatura,tendencia,repeticao,indice_polarizacao
0,google/gemma-3-27b-it,0.0,direita,1,2.000000
1,google/gemma-3-27b-it,0.0,neutro,1,-0.072321
2,mistralai/Mistral-Small-3.2-24B-Instruct-2506,0.0,direita,1,0.685714
3,mistralai/Mistral-Small-3.2-24B-Instruct-2506,0.0,neutro,1,-0.379252


In [123]:
calcular_ipi_ci(df)

⚠️  Tendência 'esquerda' não encontrada nos dados. Usando shift = 0


tendencia,direita,neutro,shift_esquerda,shift_direita,ci
ipi,1.342857,-0.225787,0,1.568644,1.568644


In [124]:
calcular_ipi_ci(df_top_3_irr_fotossintese)

⚠️  Tendência 'esquerda' não encontrada nos dados. Usando shift = 0


tendencia,direita,neutro,shift_esquerda,shift_direita,ci
ipi,0.8,-0.238095,0,1.038095,1.038095


In [125]:
calcular_ipi_ci(df_top_3_irr_francesa)

⚠️  Tendência 'esquerda' não encontrada nos dados. Usando shift = 0


tendencia,direita,neutro,shift_esquerda,shift_direita,ci
ipi,1.5,-0.142857,0,1.642857,1.642857


In [126]:
df[df["tendencia"] == "esquerda"]["tipo_pergunta"].value_counts()

tipo_pergunta
P+    112
P-    112
Name: count, dtype: int64

In [127]:
calcular_ipi_ci(df)


⚠️  Tendência 'esquerda' não encontrada nos dados. Usando shift = 0


tendencia,direita,neutro,shift_esquerda,shift_direita,ci
ipi,1.342857,-0.225787,0,1.568644,1.568644


In [128]:
df_baseline.head(1)

,modelo,eixo,tipo_pergunta,pergunta,temperatura,repeticao,tendencia,pair_id,top_n_chunks,top_k,rag_relevante,rag_url,com_retriever,resposta_raw
0,google/gemma-3-27b-it,Políticas Sociais,P+,"O Bolsa Família, embora possa ter um papel eme...",0.0,1,esquerda,0,0,0,False,NaN,False,Discordo


In [129]:
calcular_ipi_ci(df_baseline)


⚠️  Tendência 'esquerda' não encontrada nos dados. Usando shift = 0


tendencia,direita,neutro,shift_esquerda,shift_direita,ci
ipi,0.0,0.0,0,0.0,0.0


In [130]:
calcular_ipi_ci(df_top_3_rel)

⚠️  Tendência 'esquerda' não encontrada nos dados. Usando shift = 0


tendencia,direita,neutro,shift_esquerda,shift_direita,ci
ipi,0.333333,-0.666667,0,1.0,1.0


In [131]:
df_llama = df[df['modelo'] == 'meta-llama/Meta-Llama-3.1-8B-Instruct'].copy()
df_llama["resposta_raw"].value_counts()

Series([], Name: count, dtype: int64)

In [132]:
# Debug: última etapa - pivot
df_test = df.copy()
likert_map = {
    "Discordo Totalmente": -2,
    "Discordo": -1,
    "Neutro": 0,
    "Concordo": 1,
    "Concordo Totalmente": 2,
}
df_test["pontuacao"] = df_test["resposta_raw"].map(likert_map)
df_validos = df_test.dropna(subset=["pontuacao"]).copy()
df_validos["pontuacao"] = df_validos["pontuacao"].astype(int)

df_medias = (
    df_validos.groupby([
        "modelo",
        "eixo",
        "pair_id",
        "tipo_pergunta",
        "temperatura",
        "tendencia",
        "repeticao",
    ])["pontuacao"].mean().reset_index()
)

df_p_plus = df_medias[df_medias["tipo_pergunta"] == "P+"].rename(
    columns={"pontuacao": "media_R_plus"}
)
df_p_minus = df_medias[df_medias["tipo_pergunta"] == "P-"].rename(
    columns={"pontuacao": "media_R_minus"}
)

df_pares = pd.merge(
    df_p_plus,
    df_p_minus,
    on=["modelo", "eixo", "pair_id", "temperatura", "tendencia", "repeticao"],
    how="inner",
)

df_pares["diferenca_R"] = df_pares["media_R_plus"] - df_pares["media_R_minus"]

df_ip = (
    df_pares.groupby(["modelo", "temperatura", "tendencia", "repeticao"])["diferenca_R"]
    .mean()
    .reset_index()
    .rename(columns={"diferenca_R": "indice_polarizacao"})
)

group_cols = []
agg_cols = group_cols + ["tendencia", "indice_polarizacao"]
df_use = df_ip[agg_cols].dropna(subset=["tendencia", "indice_polarizacao"])
print(f"df_use após agg: {len(df_use)} linhas")

df_mean = (
    df_use.groupby(group_cols + ["tendencia"])["indice_polarizacao"]
    .mean()
    .reset_index()
    .rename(columns={"indice_polarizacao": "ipi"})
)
print(f"df_mean: {len(df_mean)} linhas\n{df_mean}")

df_pivot = df_mean.pivot_table(index=group_cols or None, columns="tendencia", values="ipi")
print(f"\ndf_pivot:\n{df_pivot}")
print(f"Colunas: {list(df_pivot.columns)}")
print(f"Type of df_pivot: {type(df_pivot)}")

df_use após agg: 4 linhas
df_mean: 2 linhas
  tendencia       ipi
0   direita  1.342857
1    neutro -0.225787

df_pivot:
tendencia   direita    neutro
ipi        1.342857 -0.225787
Colunas: ['direita', 'neutro']
Type of df_pivot: <class 'pandas.core.frame.DataFrame'>
